In [2]:
# Proof of concept and error-check for three component EMMA uncertainty module
# which is based on Genereux 1998 "Uncertainty in Tracer-Based Hydrograph Separations" (and equations in the 2022 addendum)
# Genereux (1998) uses a streamwater sample with 18O and Cl tracer data from Bazemore et al. (1994) to demonstrate
# a 3-component separation error propagation.
# Here, I make confirm I get the same uncertainties as Genereux 1998 using his Table 3 input (from Bazemore) and my python module.

import importlib
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA uncertainty propagation module

import EMMA.genereux_uncertainty_propagation as gup
importlib.reload(gup)

# -------------------------------------------------------------------
# 2. INPUT DATA FROM GENEREUX (1998) TABLE 3
# Tracer A = d18O (per mil), Tracer B = Cl (uM)
# -------------------------------------------------------------------

# Streamwater sample (Peak flow)
A_S = -6.7  # d18O
B_S = 23.0  # Cl

# End-member 1: Event water
A_1, B_1 = -8.1, 4.0

# End-member 2: Preevent soil water
A_2, B_2 = -6.1, 26.6

# End-member 3: Preevent groundwater
A_3, B_3 = -7.6, 26.0

# -------------------------------------------------------------------
# 3. DEFINE UNCERTAINTIES (W = t * sigma) AT 70% CONFIDENCE
# -------------------------------------------------------------------
# Streamwater analytical precision (df = inf, t_0.70 = 1.036)
W_AS = 1.036 * 0.19  # 0.20
W_BS = 1.036 * 0.7   # 0.73

# Component 1 (Event water): n = 8, df = 7, t_0.70 = 1.119
W_A1 = 1.119 * 0.16  # 0.18
W_B1 = 1.119 * 0.8   # 0.90

# Component 2 (Preevent soil water): n = 11, df = 10, t_0.70 = 1.093
W_A2 = 1.093 * 0.2   # 0.22
W_B2 = 1.093 * 8.3   # 9.07

# Component 3 (Preevent groundwater)
# d18O: n = 31, df = 30, t_0.70 = 1.055
# Cl:   n = 54, df = 53, t_0.70 = 1.047
W_A3 = 1.055 * 0.26  # 0.27
W_B3 = 1.047 * 3.6   # 3.77

# Assemble parameter uncertainty vector [W_AS, W_A1, W_A2, W_A3, W_BS, W_B1, W_B2, W_B3]
W_vector = np.array([W_AS, W_A1, W_A2, W_A3, W_BS, W_B1, W_B2, W_B3])

# -------------------------------------------------------------------
# 4. EXECUTE CALCULATIONS USING PYTHON MODULE
# -------------------------------------------------------------------
fractions, derivatives = gup.compute_genereux2022_fractions_and_derivatives(
    A_S, B_S, A_1, B_1, A_2, B_2, A_3, B_3
)

f1, f2, f3 = fractions

# Genereux (2022) Equations 29, 30, 31
W_f1 = np.sqrt(np.sum((np.array(derivatives['f1']) * W_vector) ** 2))
W_f2 = np.sqrt(np.sum((np.array(derivatives['f2']) * W_vector) ** 2))
W_f3 = np.sqrt(np.sum((np.array(derivatives['f3']) * W_vector) ** 2))

# -------------------------------------------------------------------
# 5. DISPLAY RESULTS AND COMPARE WITH TABLE 4
# -------------------------------------------------------------------
print("\nRESULTS FROM PYTHON MODULE:")
print(f"Component 1 (Event water)       f1: {f1:.2f} +/- {W_f1:.2f}  (Exact: {f1:.4f} +/- {W_f1:.4f})")
print(f"Component 2 (Soil water)        f2: {f2:.2f} +/- {W_f2:.2f}  (Exact: {f2:.4f} +/- {W_f2:.4f})")
print(f"Component 3 (Groundwater)       f3: {f3:.2f} +/- {W_f3:.2f}  (Exact: {f3:.4f} +/- {W_f3:.4f})")

print("\nEXPECTED VALUES FROM GENEREUX (1998) TABLE 4:")
print("Component 1 (Event water)       f1: 0.15 +/- 0.28")
print("Component 2 (Soil water)        f2: 0.65 +/- 0.19")
print("Component 3 (Groundwater)       f3: 0.20 +/- 0.41")
print("=" * 65)


RESULTS FROM PYTHON MODULE:
Component 1 (Event water)       f1: 0.15 +/- 0.28  (Exact: 0.1541 +/- 0.2753)
Component 2 (Soil water)        f2: 0.65 +/- 0.19  (Exact: 0.6514 +/- 0.1917)
Component 3 (Groundwater)       f3: 0.19 +/- 0.41  (Exact: 0.1945 +/- 0.4057)

EXPECTED VALUES FROM GENEREUX (1998) TABLE 4:
Component 1 (Event water)       f1: 0.15 +/- 0.28
Component 2 (Soil water)        f2: 0.65 +/- 0.19
Component 3 (Groundwater)       f3: 0.20 +/- 0.41
